## RAVENSTACK PROJECT
### CLEANING AND PREPROCESSING DATA TASK

**By: Chaker Saidi**

In [39]:
#import os
#import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
#import warnings
#from helper import *
#import numpy as np
# ignore warning
##warnings.filterwarnings("ignore")

# make sure printed values are not truncated
##pd.set_option('display.max_rows', None)



In [41]:
import sys
from pathlib import Path

# ---------------------------------------------------
# Ensure project root is in Python path (important for notebooks in subfolders)
# ---------------------------------------------------
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

# ---------------------------------------------------
# Standard imports
# ---------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from src.db import get_connection
from src.helper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_rows', None)

# ---------------------------------------------------
# DB connection
# ---------------------------------------------------
conn = get_connection()

In [4]:
#added conn inside brackets
accounts_df = load_accounts(conn)
events_df = load_events(conn)
support_tickets_df = load_support_tickets(conn)
feature_usage_df = load_feature_usage(conn)
subscription_df = load_subscriptions(conn)

## Account Table

In [5]:
accounts_df.head()

,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [6]:
accounts_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   account_id       500 non-null    str   
 1   account_name     500 non-null    str   
 2   industry         500 non-null    str   
 3   country          500 non-null    str   
 4   signup_date      500 non-null    object
 5   referral_source  500 non-null    str   
 6   plan_tier        500 non-null    str   
 7   seats            500 non-null    int64 
 8   is_trial         500 non-null    bool  
 9   churn_flag       500 non-null    bool  
dtypes: bool(2), int64(1), object(1), str(6)
memory usage: 32.4+ KB


In [7]:
accounts_df.describe()

,seats
count,500.000000
mean,20.560000
std,21.044718
min,1.000000
25%,5.000000
50%,15.000000
75%,28.000000
max,163.000000


In [8]:
# convert the date column to the appropriate type
accounts_df["signup_date"] = pd.to_datetime(accounts_df["signup_date"])

# check for duplicates
duplicate_total = accounts_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# check for account_id uniqueness
unique_id = (accounts_df.account_id.nunique() / len(accounts_df)) * 100
print(f"The account ids are {int(unique_id)} % unique")

# check for the earliest and latest date in the table
min_date = accounts_df["signup_date"].min()
max_date = accounts_df["signup_date"].max()
print(f"The earliest date in the account table is: {min_date}\nThe latest date in the account table is : {max_date}")


There are 0 duplicate(s) in the table
The account ids are 100 % unique
The earliest date in the account table is: 2023-01-02 00:00:00
The latest date in the account table is : 2024-12-31 00:00:00


In [9]:

# check for typos and related issues in the account table columns
concerned_cols = ["industry", "country", "referral_source", "plan_tier"]
for col in concerned_cols:
    result = accounts_df[col].value_counts()
    print('--' * 20)
    print(result.to_string())
    print('--' * 20)

----------------------------------------
industry
DevTools         113
FinTech          112
Cybersecurity    100
HealthTech        96
EdTech            79
----------------------------------------
----------------------------------------
country
US    291
UK     58
IN     49
AU     32
DE     25
CA     23
FR     22
----------------------------------------
----------------------------------------
referral_source
organic    114
other      103
ads         98
event       96
partner     89
----------------------------------------
----------------------------------------
plan_tier
Pro           178
Basic         168
Enterprise    154
----------------------------------------


## Events Table

In [10]:
events_df.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [11]:
events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    str    
 1   account_id                600 non-null    str    
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    str    
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    str    
dtypes: bool(3), float64(1), object(1), str(4)
memory usage: 30.0+ KB


In [12]:
events_df.describe()

,refund_amount_usd
count,600.000000
mean,14.420417
std,39.224591
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,392.920000


- Notice the difference between the mean 39.22 and the median 0.00 ==> skewed data !!

In [13]:
# check for missing values
miss_total = events_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = events_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (events_df.churn_event_id.nunique() / len(events_df)) * 100
print(f"The churn_event_id are {int(unique_id)} % unique")


# convert the date column
events_df["churn_date"] = pd.to_datetime(events_df["churn_date"])
min_date = events_df["churn_date"].min()
max_date = events_df["churn_date"].max()
print(f"The earliest date in the account table is: {min_date}\nThe latest date in the account table is : {max_date}")


The table contains 148 missing value(s).
There are 0 duplicate(s) in the table
The churn_event_id are 100 % unique
The earliest date in the account table is: 2023-01-25 00:00:00
The latest date in the account table is : 2024-12-31 00:00:00


- The `feedback_text` column contains 148 missing value.In the description of the column it is said that comments are optional so we can treat those missing values as 'no comment'.

- We can use this code: 

```python
    events_df["feedback_text"] = events_df["feedback_text"].fillna("no comment")
``` 

In [14]:
# check for typos and other issues in the object type columns
events_df["reason_code"].value_counts()

reason_code
features      114
support       104
budget        104
unknown        95
competitor     92
pricing        91
Name: count, dtype: int64

## Support Tickets Table

In [15]:
support_tickets_df.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.0,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.0,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False


In [16]:
support_tickets_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ticket_id                    2000 non-null   str           
 1   account_id                   2000 non-null   str           
 2   submitted_at                 2000 non-null   datetime64[us]
 3   closed_at                    2000 non-null   datetime64[us]
 4   resolution_time_hours        2000 non-null   float64       
 5   priority                     2000 non-null   str           
 6   first_response_time_minutes  2000 non-null   int64         
 7   satisfaction_score           1175 non-null   float64       
 8   escalation_flag              2000 non-null   bool          
dtypes: bool(1), datetime64[us](2), float64(2), int64(1), str(3)
memory usage: 127.1 KB


In [17]:
support_tickets_df.describe()

,submitted_at,closed_at,resolution_time_hours,first_response_time_minutes,satisfaction_score
count,2000,2000,2000.000000,2000.000000,1175.000000
mean,2024-01-05 14:08:09.600000,2024-01-07 01:59:49.200000,35.861000,88.480000,3.981277
min,2023-01-02 00:00:00,2023-01-03 03:00:00,1.000000,1.000000,3.000000
25%,2023-07-09 18:00:00,2023-07-10 21:00:00,17.000000,43.000000,3.000000
50%,2024-01-06 12:00:00,2024-01-08 05:00:00,35.000000,88.000000,4.000000
75%,2024-07-09 06:00:00,2024-07-10 11:00:00,54.000000,131.000000,5.000000
max,2024-12-31 00:00:00,2024-12-31 19:00:00,72.000000,180.000000,5.000000
std,NaN,NaN,21.138427,51.531877,0.809646


In [18]:
# check for missing values
miss_total = support_tickets_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = support_tickets_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (support_tickets_df.ticket_id.nunique() / len(support_tickets_df)) * 100
print(f"The ticket_id are {int(unique_id)} % unique")


# convert the date column
sub_min_date = support_tickets_df["submitted_at"].min()
sub_max_date = support_tickets_df["submitted_at"].max()

clos_min_date = support_tickets_df["closed_at"].min()
clos_max_date = support_tickets_df["closed_at"].max()

print(f"The earliest submission date in the support ticket table is: {sub_min_date} and the latest is : {sub_max_date}")
print(f"The earliest closed date in the support ticket table is: {clos_min_date} and the latest is : {clos_max_date}")


The table contains 825 missing value(s).
There are 0 duplicate(s) in the table
The ticket_id are 100 % unique
The earliest submission date in the support ticket table is: 2023-01-02 00:00:00 and the latest is : 2024-12-31 00:00:00
The earliest closed date in the support ticket table is: 2023-01-03 03:00:00 and the latest is : 2024-12-31 19:00:00


In [19]:
# check for typos in priority column
support_tickets_df["priority"].value_counts()

priority
urgent    514
high      510
medium    491
low       485
Name: count, dtype: int64

## Subscription Table

In [20]:
subscription_df.head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786.0,33432.0,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,None,Pro,17,833.0,9996.0,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,None,Enterprise,62,0.0,0.0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995.0,11940.0,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,None,Enterprise,27,5373.0,64476.0,False,False,False,False,monthly,True


In [21]:
subscription_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   subscription_id    5000 non-null   str    
 1   account_id         5000 non-null   str    
 2   start_date         5000 non-null   object 
 3   end_date           486 non-null    object 
 4   plan_tier          5000 non-null   str    
 5   seats              5000 non-null   int64  
 6   mrr_amount         5000 non-null   float64
 7   arr_amount         5000 non-null   float64
 8   is_trial           5000 non-null   bool   
 9   upgrade_flag       5000 non-null   bool   
 10  downgrade_flag     5000 non-null   bool   
 11  churn_flag         5000 non-null   bool   
 12  billing_frequency  5000 non-null   str    
 13  auto_renew_flag    5000 non-null   bool   
dtypes: bool(5), float64(2), int64(1), object(2), str(4)
memory usage: 376.1+ KB


In [22]:
subscription_df.describe()

,seats,mrr_amount,arr_amount
count,5000.000000,5000.000000,5000.000000
mean,29.852000,2267.749400,27212.992800
std,23.089771,3421.375348,41056.504178
min,1.000000,0.000000,0.000000
25%,14.000000,285.000000,3420.000000
50%,24.000000,931.000000,11172.000000
75%,40.000000,2786.000000,33432.000000
max,189.000000,33830.000000,405960.000000


- Notice the difference between mean and median for both `mrr_amount` and `arr_amount` columns

In [23]:
# check for missing values
miss_total = subscription_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = subscription_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (subscription_df.subscription_id.nunique() / len(subscription_df)) * 100
print(f"The subscription_id are {int(unique_id)} % unique")


# convert the date column
subscription_df["start_date"] = pd.to_datetime(subscription_df["start_date"])
subscription_df["end_date"] = pd.to_datetime(subscription_df["end_date"])
start_min_date = subscription_df["start_date"].min()
start_max_date = subscription_df["start_date"].max()

end_min_date = subscription_df["end_date"].min()
end_max_date = subscription_df["end_date"].max()

print(f"The earliest  start date in the subscription table is: {min_date} and the latest date is : {max_date}")
print(f"The earliest  end date in the subscription table is: {min_date} the latest date is : {max_date}")


The table contains 4514 missing value(s).
There are 0 duplicate(s) in the table
The subscription_id are 100 % unique
The earliest  start date in the subscription table is: 2023-01-25 00:00:00 and the latest date is : 2024-12-31 00:00:00
The earliest  end date in the subscription table is: 2023-01-25 00:00:00 the latest date is : 2024-12-31 00:00:00


- The missing values in the `end_date` are normal since they represent clients that still have valid subscription.

In [24]:
concerned_cols = ["plan_tier", "billing_frequency"]

for col in concerned_cols:
    result = subscription_df[col].value_counts()
    print('--' * 20)
    print(result.to_string())
    print('--' * 20)

----------------------------------------
plan_tier
Enterprise    1723
Pro           1675
Basic         1602
----------------------------------------
----------------------------------------
billing_frequency
monthly    2539
annual     2461
----------------------------------------


## Feature Usage Table

In [25]:
feature_usage_df.head()

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
0,U-1c6c24,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False
1,U-f07cb8,S-c25263,2023-08-07,feature_5,9,369,0,False
2,U-096807,S-f29e7f,2023-12-07,feature_3,9,1458,0,False
3,U-6b1580,S-be655e,2024-07-28,feature_40,5,2085,0,False
4,U-720a29,S-f9b1d0,2024-12-02,feature_12,12,900,0,False


In [26]:
feature_usage_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   usage_id             25000 non-null  str   
 1   subscription_id      25000 non-null  str   
 2   usage_date           25000 non-null  object
 3   feature_name         25000 non-null  str   
 4   usage_count          25000 non-null  int64 
 5   usage_duration_secs  25000 non-null  int64 
 6   error_count          25000 non-null  int64 
 7   is_beta_feature      25000 non-null  bool  
dtypes: bool(1), int64(3), object(1), str(3)
memory usage: 1.4+ MB


In [27]:
feature_usage_df.describe()

,usage_count,usage_duration_secs,error_count
count,25000.000000,25000.000000,25000.000000
mean,10.021000,3042.202880,0.564280
std,3.143729,2056.544615,1.012595
min,0.000000,0.000000,0.000000
25%,8.000000,1350.000000,0.000000
50%,10.000000,2760.000000,0.000000
75%,12.000000,4400.000000,1.000000
max,26.000000,12696.000000,8.000000


In [28]:
# check for missing values
miss_total = feature_usage_df.isna().sum().sum()
print(f"The table contains {miss_total} missing value(s).")

# check for duplicates
duplicate_total = feature_usage_df.duplicated().sum()
print(f"There are {duplicate_total} duplicate(s) in the table")

# chech for uniqueness in the churn_event_id column
unique_id = (feature_usage_df.usage_id.nunique() / len(feature_usage_df)) * 100
print(f"The usage_id are {int(unique_id)} % unique")


# convert the date column
feature_usage_df["usage_date"] = pd.to_datetime(feature_usage_df["usage_date"])

min_date = feature_usage_df["usage_date"].min()
max_date = feature_usage_df["usage_date"].max()

print(f"The earliest date in the feature usage table is: {min_date}\nThe latest date in the feature usage table is : {max_date}")


The table contains 0 missing value(s).
There are 0 duplicate(s) in the table
The usage_id are 99 % unique
The earliest date in the feature usage table is: 2023-01-01 00:00:00
The latest date in the feature usage table is : 2024-12-31 00:00:00


In [29]:

def clean_account(df, path= "..\data\processed", save= False):
    data = df.copy()
    if save:
        file_path = os.path.join(path, "processed_account.csv")
        data.to_csv(file_path, index=False)  # index=False reseting the index
        print(f"data saved in {file_path} file")
    else:
        pass
    return data
    

# Usage
clean_account(accounts_df).head()


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


In [30]:
def clean_events(df, path= "..\data\processed", save= False):
    """
    The function takes the dataframe (events) , fill in the missing values and returns a clean csv file
    """
    data = df.copy()
    data["feedback_text"] = data["feedback_text"].fillna("No Comments")
    if save:
        file_path = os.path.join(path, "processed_events.csv")
        data.to_csv(file_path, index=False)  # index=False reseting the index
        print(f"data saved in {file_path} file")
    else:
        pass
    return data

clean_events(events_df, save=False).head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,No Comments
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [31]:

def clean_subscription(df, path="..\data\processed",save= False):
    """

    I decided to keep end_date as null for active subscriptions (structural missing).

    """
    data = df.copy()
    if save:
        file_path = os.path.join(path, "processed_subscription.csv")
        data.to_csv(file_path, index=False)  # index=False reseting the index
        print(f"data saved in {file_path} file")
    else:
        pass
    return data

clean_subscription(subscription_df).head()

,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786.0,33432.0,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaT,Pro,17,833.0,9996.0,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaT,Enterprise,62,0.0,0.0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995.0,11940.0,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaT,Enterprise,27,5373.0,64476.0,False,False,False,False,monthly,True


In [32]:
support_tickets_df.sample(10)

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
1951,T-db932a,A-9779ad,2024-10-28,2024-10-28 10:00:00,10.0,urgent,92,NaN,False
118,T-ef7c6e,A-f446b6,2024-05-14,2024-05-14 01:00:00,1.0,high,47,5.0,False
144,T-b8e4c5,A-ef84cf,2023-11-03,2023-11-04 10:00:00,34.0,low,98,NaN,False
254,T-4d0d5b,A-5a215a,2024-12-06,2024-12-06 12:00:00,12.0,urgent,126,4.0,False
1788,T-dd8c7f,A-439b2f,2024-01-23,2024-01-24 06:00:00,30.0,urgent,66,NaN,False
1275,T-5bca42,A-0532a9,2023-02-11,2023-02-11 18:00:00,18.0,low,62,NaN,False
51,T-b6d7ac,A-a0ca4e,2023-10-29,2023-10-29 11:00:00,11.0,low,12,5.0,True
236,T-d9e588,A-bbe56f,2024-08-01,2024-08-01 07:00:00,7.0,low,117,3.0,False
361,T-ad6a24,A-7e0a10,2024-12-18,2024-12-19 22:00:00,46.0,urgent,43,NaN,False
150,T-dd45ee,A-8ed5dd,2024-12-09,2024-12-09 08:00:00,8.0,high,97,3.0,False


In [33]:
x = support_tickets_df[support_tickets_df["satisfaction_score"].isna()]
x["escalation_flag"].value_counts()

escalation_flag
False    793
True      32
Name: count, dtype: int64

In [34]:
x.head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,NaN,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,NaN,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,NaN,False
5,T-90f06d,A-94c3cd,2023-07-27,2023-07-27 09:00:00,9.0,medium,60,NaN,False
8,T-7119c9,A-b179bf,2023-08-27,2023-08-27 16:00:00,16.0,urgent,154,NaN,False


In [35]:
# calculate the average score among 2 groups (according to whether the support ticket was escalated or not) --> False(3.98) True(4.08)
# took the average of both score to use it to fill the missing values --> 4.03 
support_tickets_df.groupby("escalation_flag")["satisfaction_score"].mean().mean().round(2)

np.float64(4.03)

In [36]:
def clean_tickets(df, path= "..\data\processed", save = False):
    data = df.copy()
    x_mean = data.groupby("escalation_flag")["satisfaction_score"].mean().mean().round(2)
    data["satisfaction_score"] = data["satisfaction_score"].fillna(x_mean)
    if save: 
        file_path = os.path.join(path, "processed_support_tickets.csv")
        data.to_csv(file_path, index=False)  # index=False reseting the index
        print(f"data saved in {file_path} file")
    else : 
        pass
    return data

clean_tickets(support_tickets_df).head()

,ticket_id,account_id,submitted_at,closed_at,resolution_time_hours,priority,first_response_time_minutes,satisfaction_score,escalation_flag
0,T-0024de,A-712f1c,2023-07-27,2023-07-28 03:00:00,27.0,high,74,4.03,False
1,T-4d04b9,A-e43bf7,2024-07-08,2024-07-09 03:00:00,27.0,urgent,144,4.03,False
2,T-d5e12f,A-0f3e88,2024-10-17,2024-10-17 19:00:00,19.0,urgent,93,4.00,False
3,T-dfce9a,A-4c56c9,2024-09-08,2024-09-09 23:00:00,47.0,medium,126,5.00,False
4,T-c59f77,A-6f8ad2,2024-11-30,2024-12-01 02:00:00,26.0,medium,8,4.03,False


In [38]:
def clean_feature_usage(df, save=False, keep_key=True):
    data = df.copy()

    if not keep_key:
        # Drop usage_id and create surrogate key
        data = data.drop(columns=["usage_id"])
        data["row_id"] = range(len(data))

    if save:

        #-- file_path = os.path.join(path, "processed_feature_usage.csv")
        #-- the above line for "file_path" is commented out and replaced.
        #-- path was defined explicitly above as path="../data/processed" which resulted in
        #-- file_path = "../data/processed/" = 
        #--> but "notebooks/data/processed" folder does not exist

        # Use an anchor to the project root to solve the problem
        # CHANGED: we no longer pass `path` as a function argument
        # (see function argument above) because
        # paths are now centrally managed in src/config.py (our single source of truth)

        from src.config import PROCESSED_DIR

        # CHANGED: ensure output directory exists using pathlib (consistent with project style)
        PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

        # CHANGED: file path is now built from project-wide PROCESSED_DIR
        file_path = PROCESSED_DIR / "processed_feature_usage.csv"

        if file_path.exists():
            response = input(f"{file_path} already exists. Overwrite? (y/n): ")

            if response.lower() != "y":
                print("Save cancelled.")
                return data

        data.to_csv(file_path, index=False)

        if keep_key:
            print(f"Data saved in {file_path}. Recommendation: use composite index to work with the table.")
        else:
            print(f"Data saved in {file_path} with a new surrogate id column.")



    return data


clean_feature_usage(feature_usage_df, save=True, keep_key=False).head()

Save cancelled.


,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature,row_id
0,S-0fcf7d,2023-07-27,feature_20,9,5004,0,False,0
1,S-c25263,2023-08-07,feature_5,9,369,0,False,1
2,S-f29e7f,2023-12-07,feature_3,9,1458,0,False,2
3,S-be655e,2024-07-28,feature_40,5,2085,0,False,3
4,S-f9b1d0,2024-12-02,feature_12,12,900,0,False,4


**The data is ready for the next step**